In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

CATALOG = "gwd_de"
source_table = f"{CATALOG}.bronze.invoice_line_items"
target_table = f"{CATALOG}.silver.invoice_line_items"

In [0]:
df_bronze = spark.table(source_table)

df_silver = df_bronze.select(
    F.col("invoice_line_id"),
    F.col("invoice_id"),
    F.col("product_id"),
    F.col("quantity"),
    F.coalesce(F.col("unit_price").cast(DecimalType(10, 2)), F.lit(0.00)).alias("unit_price"),
    F.coalesce(F.col("discount"), F.lit(0.00)).alias("discount")
)

df_silver = df_silver.withColumn(
    "line_total",
    (F.col("quantity") * F.col("unit_price") * (F.lit(1) - F.col("discount"))).cast(DecimalType(10, 2))
)

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)